# DDRM — Diffusion Restoration for Beam-Erased Kinematics

Trains an **unconditional** diffusion prior over clean disk channel maps, then restores the
self-gravitating dirty cube with DDRM using the measured beam operator.

**Why:** the beam alone erases the GI wiggle (residual RMS 1.394 → 0.174, r=0.116 vs truth).
A denoiser cannot undo that. DDRM can, in principle, because it reconstructs what the
instrument could not measure.

**Target to beat:** wiggle residual correlation 0.116 (beam-only floor). Above that is real
recovery; at that level the prior is hallucinating.

## 0. Bootstrap

In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    hits = glob.glob('/kaggle/input/**/*dirty*.fits', recursive=True)
    if not hits:
        raise FileNotFoundError(
            'No dirty FITS under /kaggle/input. Attach the line-emission Dataset.\n'
            f'Attached: {sorted(glob.glob("/kaggle/input/*")) or "(nothing)"}')
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0]))

    sg = glob.glob('/kaggle/input/**/clean_sg.fits', recursive=True)
    SG_DIR = os.path.dirname(sg[0]) if sg else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../Line Emission Data'
    SG_DIR = '../self-gravitating cube and dirty cube/kinematic_data_v2'

print('cwd:', os.getcwd())
print('DATA_DIR:', DATA_DIR)
print('SG_DIR  :', SG_DIR or 'NOT FOUND — section 5 will be skipped')

## 0b. Pull latest `src/` (no kernel restart)

In [ ]:
import importlib, subprocess, sys
if ON_KAGGLE:
    subprocess.run(['git','-C','/kaggle/working/EXXA','fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C','/kaggle/working/EXXA','reset','--hard','origin/'+BRANCH], check=True)
    print(subprocess.run(['git','-C','/kaggle/working/EXXA','log','-1','--oneline'],
                         capture_output=True, text=True).stdout.strip())

for m in [m for m in list(sys.modules) if m.startswith('src.')]:
    importlib.reload(sys.modules[m])
print('src/ reloaded')

## 1. Config

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

TARGET_SIZE = 256
N_SAMPLES   = 150      # channels per cube
BATCH_SIZE  = 8
EPOCHS      = 60
LR          = 2e-5
PREDICTION  = 'v'      # v + min_snr_gamma=5 converged best in notebook 06
MIN_SNR     = 5.0

CKPT = '../results/checkpoints/ddrm_prior.pth'
os.makedirs(os.path.dirname(CKPT), exist_ok=True)

print(f'device: {device} | GPUs: {N_GPU}')
print(f'{TARGET_SIZE}px | batch {BATCH_SIZE} | {EPOCHS} epochs | {PREDICTION}-prediction')

## 2. Data — clean channel maps only

The prior is over clean images. `FITSChannelDataset` returns `(dirty, clean)`; we keep the
clean half. Same Jy/beam domain as `clean_sg.fits`.

In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(
    data_dir=DATA_DIR, n_holdout=3, val_fraction=0.2, seed=SEED)

_ds_kw = dict(n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
              subtract_continuum=True, continuum_n=5, verbose=False)
train_pairs = FITSChannelDataset(train_cubes, **_ds_kw)
val_pairs   = FITSChannelDataset(val_cubes, **_ds_kw)


class CleanOnly(torch.utils.data.Dataset):
    """(dirty, clean) -> clean. The prior never sees a dirty image."""
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i): return self.pairs[i][1]


train_ds, val_ds = CleanOnly(train_pairs), CleanOnly(val_pairs)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(N_GPU > 0), drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'train {len(train_ds)} | val {len(val_ds)} images')
_x = train_ds[0]
print(f'sample: {tuple(_x.shape)}  range [{_x.min():.3f}, {_x.max():.3f}]')
assert _x.shape[0] == 1, 'prior expects single-channel clean images'

## 3. Train the unconditional prior

In [ ]:
from src.models.diffusion_unet import default_diffusion_config
from src.training.diffusion import DenoisingDiffusion

cfg = default_diffusion_config(image_size=TARGET_SIZE)
cfg.data.conditional = False           # p(clean), not p(clean|dirty)
cfg.diffusion.prediction_type = PREDICTION
cfg.diffusion.min_snr_gamma = MIN_SNR
cfg.diffusion.beta_schedule = 'cosine'

runner = DenoisingDiffusion(config=cfg, device=str(device), lr=LR, checkpoint_path=CKPT)
print(f'params: {sum(p.numel() for p in runner._core.parameters())/1e6:.1f}M')

t0 = time.time()
history = runner.train(train_loader, val_loader, n_epochs=EPOCHS, verbose=True)
print(f'\ntrained in {(time.time()-t0)/60:.1f} min -> {CKPT}')

In [ ]:
# Persist immediately (RULES.md #1) — CKPT_DIR is inside the git clone the bootstrap wipes.
import shutil
if ON_KAGGLE:
    out = '/kaggle/working/ddrm_prior.pth'
    shutil.copy2(CKPT, out)
    print('persisted ->', out, f'({os.path.getsize(out)/1e6:.0f} MB)')

## 4. Loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history['train_losses'], label='train')
if history.get('val_losses'): ax.plot(history['val_losses'], label='val')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.set_yscale('log')
ax.legend(); ax.grid(alpha=0.3); ax.set_title('Unconditional prior')
plt.tight_layout(); plt.savefig('../results/ddrm_prior_loss.png', dpi=130)
plt.show()

## 5. DDRM restoration

Restore `dirty_sg.fits` using the prior plus the measured beam. Scored against
`clean_sg.fits` on the GI wiggle residual — the metric a plausible-looking image cannot fake.

In [ ]:
assert SG_DIR, 'self-gravitating cube not attached; cannot run section 5'

import numpy as np
from astropy.io import fits
from src.training.ddrm import beam_transfer_function, ddrm_steps
from src.training.diffusion import data_transform, inverse_data_transform

TRIM = (60, 541)
CHANNELS = list(range(280, 320, 4))     # 10 channels near the line peak

with fits.open(f'{SG_DIR}/clean_sg.fits', memmap=True) as h:
    sg_hdr = h[0].header
    sg_clean = np.stack([np.asarray(h[0].data[c], np.float32) for c in CHANNELS])
with fits.open(f'{SG_DIR}/dirty_sg.fits', memmap=True) as h:
    sg_dirty = np.stack([np.asarray(h[0].data[c], np.float32) for c in CHANNELS])

beam = fits.getdata('../results/self-gravitating/dirty_beam_recovered_v2.fits').astype(np.float64)
print(f'{len(CHANNELS)} channels | beam {beam.shape} peak {beam.max():.3f}')

In [ ]:
import torch.nn.functional as Fn

def to01(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a), lo, hi

restored, dirty_rs = [], []
runner.model.eval()
for i in range(len(CHANNELS)):
    d01, lo, hi = to01(sg_dirty[i])
    t = torch.from_numpy(d01)[None, None].float().to(device)
    t = Fn.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
    y = data_transform(t)

    transfer = beam_transfer_function(beam, (TARGET_SIZE, TARGET_SIZE), device=device)
    transfer = transfer / transfer.max()          # normalise: unit gain at DC

    sigma_y = float(sg_hdr.get('RMS', 0.0013)) / max(hi - lo, 1e-12) * 2.0

    with torch.no_grad():
        xs, _ = ddrm_steps(y, list(range(0, runner.num_timesteps, runner.num_timesteps // 50)),
                           runner.model, runner.betas, transfer, sigma_y=sigma_y,
                           prediction_type=PREDICTION)
    out = inverse_data_transform(xs[-1].to(device))
    out = Fn.interpolate(out, sg_clean.shape[-2:], mode='bilinear', align_corners=False)
    restored.append(out[0, 0].cpu().numpy() * (hi - lo) + lo)
    dirty_rs.append(sg_dirty[i])
    print(f'  channel {CHANNELS[i]} done', flush=True)

restored = np.stack(restored)
print('restored:', restored.shape)

## 6. Score — does DDRM beat the beam-only floor?

In [ ]:
from src.evaluation.moment_maps import generate_moment_maps, signal_mask
from src.evaluation.gi_wiggle import quadratic_moment1, fit_keplerian, wiggle_residual

velax = ((sg_hdr['CRVAL3'] + (np.array(CHANNELS) + 1 - sg_hdr['CRPIX3']) * sg_hdr['CDELT3'])
         * 1000.0)
AU_PER_PX = abs(sg_hdr['CDELT1']) * 3600.0 * sg_hdr.get('DIST_PC', 140.0)

m0, _, _ = generate_moment_maps('', data_velax=(sg_clean.astype(np.float64), velax))
mask = signal_mask(m0, frac=0.02)

rows = {}
for tag, cube in (('clean', sg_clean), ('dirty', sg_dirty), ('DDRM', restored)):
    v0, _ = quadratic_moment1(cube.astype(np.float64), velax)
    m1 = v0 / 1000.0
    init = None if tag == 'clean' else {k: rows['clean']['geom'][k]
                                        for k in ('cx','cy','pa_deg','incl_deg')}
    geom = fit_keplerian(m1, mask, AU_PER_PX, init=init)
    rows[tag] = dict(m1=m1, geom=geom, resid=wiggle_residual(m1, geom))

print(f"{'':8s} {'mstar':>7s} {'incl':>6s} {'raw r':>8s} {'resid r':>9s}")
for tag in ('dirty', 'DDRM'):
    ok = np.isfinite(rows['clean']['m1'][mask]) & np.isfinite(rows[tag]['m1'][mask])
    raw = np.corrcoef(rows['clean']['m1'][mask][ok], rows[tag]['m1'][mask][ok])[0, 1]
    ok2 = np.isfinite(rows['clean']['resid'][mask]) & np.isfinite(rows[tag]['resid'][mask])
    res = np.corrcoef(rows['clean']['resid'][mask][ok2], rows[tag]['resid'][mask][ok2])[0, 1]
    g = rows[tag]['geom']
    print(f"{tag:8s} {g['mstar_msun']:7.3f} {g['incl_deg']:6.1f} {raw:8.4f} {res:9.4f}")
    if tag == 'DDRM':
        print(f"\n  beam-only floor: 0.116")
        print(f"  DDRM:            {res:.4f}  ->  "
              f"{'RECOVERY' if res > 0.20 else 'no recovery beyond the floor'}")

## 7. Figure

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
vm1 = np.nanpercentile(np.abs(rows['clean']['m1'][mask]), 98)
vr  = np.nanpercentile(np.abs(rows['clean']['resid'][mask]), 98)
for i, tag in enumerate(('clean', 'dirty', 'DDRM')):
    im = ax[0, i].imshow(np.where(mask, rows[tag]['m1'], np.nan), cmap='RdBu_r',
                         vmin=-vm1, vmax=vm1, origin='lower')
    ax[0, i].set_title(f'{tag}: M1'); plt.colorbar(im, ax=ax[0, i], fraction=0.046)
    im2 = ax[1, i].imshow(np.where(mask, rows[tag]['resid'], np.nan), cmap='RdBu_r',
                          vmin=-vr, vmax=vr, origin='lower')
    ax[1, i].set_title(f'{tag}: Keplerian residual'); plt.colorbar(im2, ax=ax[1, i], fraction=0.046)
plt.suptitle('DDRM restoration — top: moment-1, bottom: GI wiggle residual')
plt.tight_layout(); plt.savefig('../results/ddrm_restoration.png', dpi=130)
plt.show()

## 8. Collect outputs

In [ ]:
from src.evaluation.collect_outputs import collect_outputs
collect_outputs('07-ddrm-restoration')